# Memory from scratch

Draft code for lesson 1.8. Each stage solves the problem the previous stage exposed:

| Stage | Problem it solves | What we build |
|---|---|---|
| 0 | (none, the baseline) | an agent with no memory |
| 1 | "You forgot what I said a second ago" | one thread: a list of messages |
| 2 | "I want separate conversations" | many threads: a dict of lists |
| 3 | "You forgot me when I started a new chat" | long-term memory: a `remember` tool and a prompt |
| 4 | "Now my roommate uses it too" | (no new code) the shared memory leaks |
| 5 | "Ben shouldn't see Ana's memories" | memory namespaced by user |
| 6 | "What do you actually know about me?" | one memory file per user, written through the filesystem MCP server |

Stages 2 and 5 end with the "few lines" framework version, so you can map it back to what we built.

In [ ]:
%pip install -q -U "langchain>=1.4" langchain-google-genai

In [ ]:
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain.tools import tool

MODEL = "google_genai:gemini-flash-lite-latest"


def show(messages):
    """Print a list of messages the way the agent sees them."""
    for m in messages:
        m.pretty_print()

## Stage 0: an agent with no memory

Every `invoke` starts from nothing. The model only sees the messages in *this* call.

In [ ]:
agent = create_agent(MODEL, tools=[])

result = agent.invoke({"messages": [HumanMessage("Hi, I'm Ana. I'm taking DS 250 and I prefer R over Python.")]})
show(result["messages"])

result = agent.invoke({"messages": [HumanMessage("What's my name, and which language do I like?")]})
show(result["messages"])

Look at what came back from the second call: one `HumanMessage` and one `AIMessage`.
The first exchange isn't there, so the model couldn't have known. **The model is stateless.**
Whatever it "remembers" is whatever you send it.

## Stage 1: one thread

So we send it everything. A conversation is just a list of messages that we keep and send again each turn.

In [ ]:
result = agent.invoke({
    "messages": [
        HumanMessage("Hi, I'm Ana. I'm taking DS 250 and I prefer R over Python."),
        AIMessage("Hi Ana! Nice to meet you. \n\nDS 250 is a great class, and R is a fantastic tool for data science (especially with the `tidyverse`!). \n\nSince I can write and debug in both languages, I'll make sure to provide R solutions and code snippets for your assignments and concepts. \n\nHow can I help you with DS 250 today? Are you working on a specific lab, homework, or project?"),
        HumanMessage("What's my name, and which language do I like?"),
    ]
})

show(result["messages"])

But that is a lot to maintain. So let's save the conversation dynamically.

In [ ]:
thread = []


def chat(text):
    thread.append(HumanMessage(text))
    result = agent.invoke({"messages": thread})
    new_messages = result["messages"][len(thread):]   # what the agent added this turn
    thread.extend(new_messages)
    return thread[-1].text


print(chat("Hi, I'm Ana. I'm taking DS 250 and I prefer R over Python."))
print(chat("What's my name, and which language do I like?"))

In [ ]:
show(thread)

That list *is* the memory. Notice it grows every turn, and we pay to resend all of it every time.

## Stage 2: many threads

Ana uses the agent for homework. The next day she comes back to start her final project, and she wants a
fresh conversation. But our app only has one list, so there's no such thing as a "new" conversation.

In [ ]:
# Ana's homework session
print(chat("Can you explain what a p-value is in one paragraph?"))
print(chat("Give me a quick example in R."))

# The next day: Ana means to start fresh on her project
print(chat("I'm starting my final project on bike-share ride data. What should I do first?"))
print(chat("Summarize what we've talked about in this conversation."))

In [ ]:
print(len(thread), "messages")
show(thread)

Ana meant this to be two conversations. The app saw one long one. Every project question resends the whole
homework session, so the model has old material to get distracted by, and Ana pays for those tokens every turn.
There's also no way to go back to just the homework.

The fix is to give each conversation a name, a **thread ID**, and keep one list per thread ID: a dict of lists.

In [ ]:
threads = {}


def chat(thread_id, text):
    history = threads.setdefault(thread_id, [])
    history.append(HumanMessage(text))
    result = agent.invoke({"messages": history})
    history.extend(result["messages"][len(history):])
    return history[-1].text


print(chat("homework", "Hi, I'm Ana. I'm taking DS 250 and I prefer R over Python."))
print(chat("homework", "Can you explain what a p-value is in one paragraph?"))
print(chat("homework", "Give me a quick example in R."))

print(chat("project", "I'm starting my final project on bike-share ride data. What should I do first?"))
print(chat("project", "Summarize what we've talked about in this conversation."))

In [ ]:
for thread_id, history in threads.items():
    print(f"{thread_id}: {len(history)} messages")

show(threads["project"])

Same messages, but now there are two lists, and the project summary only covers the project.

In [ ]:
print(chat("project", "Which language should I use for this project, given what I prefer?"))

The project thread has no idea Ana prefers R. That's what we asked for (the threads are separate), but it's
also the next problem: some things should carry over to *every* conversation.

### The framework version

What we built is what a **checkpointer** does. `InMemorySaver` keeps the dict of lists, and
`thread_id` picks the list.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

fw_agent = create_agent(MODEL, tools=[], checkpointer=InMemorySaver())

homework = {"configurable": {"thread_id": "homework"}}
project = {"configurable": {"thread_id": "project"}}

fw_agent.invoke({"messages": [HumanMessage("Hi, I'm Ana. I prefer R over Python.")]}, homework)
fw_agent.invoke({"messages": [HumanMessage("Which language do I like?")]}, homework)
fw_agent.invoke({"messages": [HumanMessage("Which language do I like?")]}, project)

# The checkpointer's saved list for one thread: the same kind of list as threads["homework"] above
show(fw_agent.get_state(homework).values["messages"])

Note: we only send the *new* message now. The checkpointer loads the old ones and saves the new ones,
which is the `append` / `extend` we wrote by hand.

## Stage 3: long-term memory

Some facts are about *Ana*, not about one conversation. They need to live outside every thread
and be put into each one. Long-term memory has two halves:

- **Write:** something has to decide a fact is worth keeping and save it. We let the model decide,
  by giving it a `remember` tool.
- **Read:** saved facts have to reach the model. We put them in the system prompt at the start of every turn.

In [ ]:
memories = []   # lives outside every thread


@tool
def remember(fact: str) -> str:
    """Save a lasting fact about the user (name, classes, preferences, goals) for future conversations."""
    memories.append(fact)
    return "Saved."


BASE_PROMPT = (
    "You are a helpful tutor. When the user tells you something lasting about themselves, "
    "call the remember tool to save it."
)


def build_system_prompt(facts):
    if not facts:
        return BASE_PROMPT
    return BASE_PROMPT + "\n\nWhat you remember about the user:\n" + "\n".join(f"- {f}" for f in facts)


memory_agent = create_agent(MODEL, tools=[remember])
threads = {}


def chat(thread_id, text):
    history = threads.setdefault(thread_id, [])
    history.append(HumanMessage(text))

    # Read: rebuild the system prompt from memory every turn. It is NOT saved in the thread.
    system = SystemMessage(build_system_prompt(memories))
    result = memory_agent.invoke({"messages": [system] + history})

    history.extend(result["messages"][len(history) + 1:])   # +1 skips the system message
    return history[-1].text


print(chat("homework", "Hi, I'm Ana. I'm taking DS 250 and I prefer R over Python."))
print(memories)

In [ ]:
print(chat("project", "Which language do I like?"))

A brand-new thread knows Ana's preference. Look at how the fact got saved: the model *chose* to call a tool,
and the tool call and its result are just two more messages in the thread.

In [ ]:
show(threads["homework"])

In [ ]:
show(threads["project"])   # no tool call here; the fact arrived through the system prompt

## Stage 4: a second user

Ana's roommate Ben starts using the same agent. Same code, nothing changed. Ben opens his own thread.

In [ ]:
print(chat("ben-1", "Hey, what do you know about me?"))

Ben gets Ana's facts. Separate threads didn't help, because `memories` was never per-thread. It's one list
for everyone.

Sometimes a shared memory is what you want (team conventions, a shop's policies). For facts about a *person*,
it's a leak.

## Stage 5: each user gets their own memory

Two changes:

1. **Memories keyed by user:** a dict of lists, `memories[user_id]`.
2. **Threads keyed by user too:** `(user_id, thread_id)`, so Ben can't open Ana's `"homework"` thread.

The tricky part is the `remember` tool. It has to write to the right user's list, but **the model must
not pick the user**. If `user_id` were a tool argument, the model could write to (or be talked into
writing to) anyone's memory. The user ID comes from the app (who logged in), so we bake it into the tool.

In [ ]:
memories = {}   # user_id -> list of facts
threads = {}    # (user_id, thread_id) -> list of messages


def make_remember_tool(user_id):
    @tool
    def remember(fact: str) -> str:
        """Save a lasting fact about the user (name, classes, preferences, goals) for future conversations."""
        memories.setdefault(user_id, []).append(fact)
        return "Saved."

    return remember


def chat(user_id, thread_id, text):
    history = threads.setdefault((user_id, thread_id), [])
    history.append(HumanMessage(text))

    user_agent = create_agent(MODEL, tools=[make_remember_tool(user_id)])
    system = SystemMessage(build_system_prompt(memories.get(user_id, [])))
    result = user_agent.invoke({"messages": [system] + history})

    history.extend(result["messages"][len(history) + 1:])
    return history[-1].text


print(chat("ana", "homework", "Hi, I'm Ana. I'm taking DS 250 and I prefer R over Python."))
print(chat("ben", "homework", "I'm Ben. I'm a Python person, and I'm in DS 350."))
print(chat("ben", "chat-2", "What do you know about me?"))
print(chat("ana", "chat-2", "What do you know about me?"))

In [ ]:
memories

In [ ]:
for key, history in threads.items():
    print(key, len(history), "messages")

### The framework version

Every piece we built has a name in LangChain:

| We built | LangChain |
|---|---|
| `threads[thread_id]` (a list per conversation) | `checkpointer` + `thread_id` |
| `memories` (outside every thread) | `store` |
| `memories[user_id]` | a store **namespace**, e.g. `("ana", "memories")` |
| `user_id` baked into the tool | `context`, read in the tool via `runtime.context` |
| rebuilding the system prompt each turn | `@dynamic_prompt` middleware |

In [ ]:
import uuid
from dataclasses import dataclass

from langchain.agents.middleware import ModelRequest, dynamic_prompt
from langchain.tools import ToolRuntime
from langgraph.store.memory import InMemoryStore


@dataclass
class Context:
    user_id: str


@tool
def remember(fact: str, runtime: ToolRuntime[Context]) -> str:
    """Save a lasting fact about the user (name, classes, preferences, goals) for future conversations."""
    namespace = (runtime.context.user_id, "memories")
    runtime.store.put(namespace, str(uuid.uuid4()), {"fact": fact})
    return "Saved."


@dynamic_prompt
def prompt_with_memories(request: ModelRequest) -> str:
    namespace = (request.runtime.context.user_id, "memories")
    facts = [item.value["fact"] for item in request.runtime.store.search(namespace)]
    return build_system_prompt(facts)


store = InMemoryStore()
fw_agent = create_agent(
    MODEL,
    tools=[remember],
    middleware=[prompt_with_memories],
    checkpointer=InMemorySaver(),
    store=store,
    context_schema=Context,
)


def fw_chat(user_id, thread_id, text):
    result = fw_agent.invoke(
        {"messages": [HumanMessage(text)]},
        {"configurable": {"thread_id": thread_id}},
        context=Context(user_id=user_id),
    )
    return result["messages"][-1].text


print(fw_chat("ana", "ana-homework", "Hi, I'm Ana. I'm taking DS 250 and I prefer R over Python."))
print(fw_chat("ben", "ben-homework", "I'm Ben. I'm a Python person, and I'm in DS 350."))
print(fw_chat("ben", "ben-chat-2", "What do you know about me?"))

In [ ]:
for user_id in ["ana", "ben"]:
    print(user_id, [item.value["fact"] for item in store.search((user_id, "memories"))])

One thing the framework does *not* do for you: the checkpointer keys threads by `thread_id` alone, not by user.
In our version, `(user_id, thread_id)` kept Ben out of Ana's threads. With the framework, your app has to make
sure a user can only use their own thread IDs (here, by prefixing them; in a real app, by checking who owns
the thread before calling the agent).

## Stage 6: one memory file per user

The store works, but Ana can't see what's in it. Each fact is a separate row in RAM. And when she changes her
mind, the store just adds another row, so it keeps both "prefers R" and "switched to Python".

Let's try something simpler: **one plain text file per user**. Anyone can open it and read it. The model can
rewrite a line instead of adding a new one. And it's still there after the Python process ends. (Claude Code's
`CLAUDE.md` file works the same way.)

The model doesn't need a hand-written `remember` tool anymore. The filesystem MCP server from Lesson 1.7
already has `write_file` and `edit_file`.

In [ ]:
%pip install -q -U "langchain[mcp]>=1.4"

In [ ]:
from langchain.mcp import MCPAdapter

MEMORY_ROOT = "/content/memories"
WRITE_TOOLS = ("write_file", "edit_file")   # the only filesystem tools the model needs here
checkpointer = InMemorySaver()
file_agents = {}   # user_id -> agent


async def get_file_agent(user_id):
    if user_id in file_agents:
        return file_agents[user_id]

    folder = f"{MEMORY_ROOT}/{user_id}"
    os.makedirs(folder, exist_ok=True)
    path = f"{folder}/memory.md"

    # The server can only touch the folders listed at the end of args: this user's folder, nothing else.
    config = {"mcpServers": {
        "memory": {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", folder]},
    }}
    async with MCPAdapter(config) as adapter:
        tools = [t for t in await adapter.list_tools() if t.name.endswith(WRITE_TOOLS)]

    @dynamic_prompt
    def prompt_with_memory_file(request: ModelRequest) -> str:
        contents = open(path).read() if os.path.exists(path) else "(empty)"
        return (
            f"You are a helpful tutor. You keep one memory file about the user at {path}. "
            "When the user tells you something lasting about themselves, update that file. "
            "Keep it a short bulleted list, and edit existing lines instead of repeating them.

"
            f"Current contents of {path}:
{contents}"
        )

    agent = create_agent(MODEL, tools=tools, middleware=[prompt_with_memory_file], checkpointer=checkpointer)
    file_agents[user_id] = agent
    return agent


async def file_chat(user_id, thread_id, text):
    agent = await get_file_agent(user_id)
    result = await agent.ainvoke(
        {"messages": [HumanMessage(text)]},
        {"configurable": {"thread_id": f"{user_id}-{thread_id}"}},
    )
    return result["messages"][-1].text

Two halves again, just split differently:

- **Write:** the model decides what's worth keeping, so it gets the MCP write tools.
- **Read:** the file has to reach the model *every* turn, so that's the app's job. `@dynamic_prompt` opens it
  with plain Python. If reading were a tool, the model could forget to check.

User isolation also works differently. In Stage 5 we baked `user_id` into the tool. Here the **server** enforces it.
Ana's agent gets a filesystem server that can only see `/content/memories/ana`, so it *can't* write to Ben's file,
no matter what anyone types.

In [ ]:
print(await file_chat("ana", "homework", "Hi, I'm Ana. I'm taking DS 250 and I prefer R over Python."))
print(await file_chat("ben", "homework", "I'm Ben. I'm a Python person, and I'm in DS 350."))
print(await file_chat("ana", "project", "Heads up: I've switched to Python this semester."))
print(await file_chat("ana", "chat-3", "What do you know about me?"))

In [ ]:
for user_id in ["ana", "ben"]:
    print(f"--- {MEMORY_ROOT}/{user_id}/memory.md")
    print(open(f"{MEMORY_ROOT}/{user_id}/memory.md").read())

Open the **Files** pane and look in `memories/`. Ana's file should say Python now, not both. Try changing the
file by hand, then ask the agent what it knows about you. Your change shows up on the next turn, because the
file *is* the memory.

What this costs:

- **The whole file goes into every prompt.** As it grows, you pay for it every turn, just like the thread in Stage 1.
- **Two chats editing at once can overwrite each other.** A database handles that; one plain file doesn't.
- **Colab wipes `/content` when the runtime resets.** On your laptop the file stays. In Colab, point
  `MEMORY_ROOT` at Google Drive if you want it to last.